# 05c: Cross-station figures

Builds figures from saved tables and instrumented score series. BOCPD is not rerun.


## Section 0: Setup and text-rendering check


In [ ]:
import json, pathlib, warnings
import numpy  as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=FutureWarning)
RNG = np.random.default_rng(20240604)




FONT_FAMILY = "DejaVu Sans"   
                                
plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white",
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,
    "font.size":10,"font.family":FONT_FAMILY,
    "axes.titlesize":11,"axes.labelsize":10,
    "legend.fontsize":8.5,"figure.titlesize":12,
})
COLOR_DISCHARGE_BOCPD = "#1a6faf"
COLOR_STATIC_P95       = "#e07b39"
COLOR_TURBIDITY_BOCPD  = "#2a9d8f"
COLOR_MAKIKI_REF       = "#888888"
METHOD_COLORS = {"discharge_bocpd": COLOR_DISCHARGE_BOCPD, "static_discharge_p95": COLOR_STATIC_P95,
                   "turbidity_bocpd": COLOR_TURBIDITY_BOCPD}
METHOD_MARKERS = {"discharge_bocpd": "o", "static_discharge_p95": "s", "turbidity_bocpd": "^"}


NOTEBOOK_DIR  = pathlib.Path(".").resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parent
RAW_DIR       = PROJECT_ROOT / "Data"    / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data"    / "Processed"
TABLE_DIR     = PROJECT_ROOT / "Outputs" / "Tables"
CONFIG_DIR    = PROJECT_ROOT / "Outputs" / "Config"
FIG_ROOT      = PROJECT_ROOT / "Outputs" / "Figures" / "05c_inventory"
FIG_DIRS = {k: FIG_ROOT / v for k, v in {
    "A": "A_data_availability", "B": "B_primary_performance", "C": "C_campaign_level",
    "D": "D_bocpd_diagnostics", "E": "E_robustness", "F": "F_static_and_turbidity",
    "G": "G_dashboards",
}.items()}
for d in [TABLE_DIR, CONFIG_DIR] + list(FIG_DIRS.values()):
    d.mkdir(parents=True, exist_ok=True)

SAVE_OUTPUTS = True
MAKIKI_ID = "16238000"
EXTERNAL_STATIONS = {"16247100": "Manoa-Palolo Drainage Canal at Moiliili",
                       "16265000": "Kawa Stream at Kaneohe",
                       "16274100": "Kaneohe Stream below Kamehameha Hwy"}
ALL_STATION_IDS = [MAKIKI_ID] + list(EXTERNAL_STATIONS.keys())
STATION_NAMES = {MAKIKI_ID: "Makiki Stream at King St. bridge", **EXTERNAL_STATIONS}

def save_fig(category, filename, fig=None, dpi=140):
    path = FIG_DIRS[category] / filename
    target_fig = fig or plt.gcf()
    try:
        target_fig.tight_layout()
    except Exception as e:
        print(f"  tight_layout failed for {filename}: {type(e).__name__}")
    target_fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.show(); plt.close("all")
    print(f"  Saved: {category}/{filename}")

def save_csv(df, path, label="", gz=False):
    path = pathlib.Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, compression="gzip" if gz else None)
    print(f"  Saved {label or path.name}  ({len(df):,} rows)")

def combined_legend(ax_primary, *other_axes, loc="upper left", fontsize=8.5):
    handles, labels = ax_primary.get_legend_handles_labels()
    for ax in other_axes:
        h, l = ax.get_legend_handles_labels()
        handles += h; labels += l
    ax_primary.legend(handles, labels, loc=loc, fontsize=fontsize)

def find_file(names, dir_, required=True):
    for n in names:
        p = pathlib.Path(dir_) / n
        if p.exists(): return p
    if required: raise FileNotFoundError(f"Missing file: {[str(pathlib.Path(dir_)/n) for n in names]}")
    return None

def wilson_ci(successes: int, n: int, z: float = 1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    phat = successes / n
    denom = 1 + z**2/n
    center = (phat + z**2/(2*n)) / denom
    half = (z/denom) * np.sqrt(phat*(1-phat)/n + z**2/(4*n**2))
    return (phat, max(0.0, center-half), min(1.0, center+half))

print("Setup complete.")
print(f"  Font family requested : {FONT_FAMILY}")
try:
    resolved = fm.findfont(FONT_FAMILY, fallback_to_default=True)
    print(f"  Font resolved to      : {resolved}")
except Exception as e:
    print(f"  Font resolution check failed: {e}")
print(f"  Matplotlib backend    : {matplotlib.get_backend()}")
print(f"  Matplotlib version    : {matplotlib.__version__}")


In [ ]:
from matplotlib.backends.backend_agg import FigureCanvasAgg

fig, ax = plt.subplots(figsize=(4,3))
ax.plot([1,2,3], [1,4,2], label="test series")
ax.set_title("Self-test title")
ax.set_xlabel("Self-test x label")
ax.set_ylabel("Self-test y label")
ax.legend()
fig.canvas.draw()

checks = {"title": ax.title, "xlabel": ax.xaxis.label, "ylabel": ax.yaxis.label}
results = {}
for name, artist in checks.items():
    bbox = artist.get_window_extent(renderer=fig.canvas.get_renderer())
    results[name] = (bbox.width > 1 and bbox.height > 1)
leg = ax.get_legend()
leg_text_artist = leg.get_texts()[0] if leg and leg.get_texts() else None
results["legend_text"] = (leg_text_artist.get_window_extent(renderer=fig.canvas.get_renderer()).width > 1) if leg_text_artist else False

print("Part 1: live renderer")
for name, ok in results.items():
    print(f"  {name:<14}: {'OK' if ok else 'FAILED'}")

# Test the Agg save path.


fig2, ax2 = plt.subplots(figsize=(4,4))
ax2.scatter([1,2,3],[1,2,1])
ax2.set_title("Save-path test title"); ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_aspect("equal")
agg_canvas = FigureCanvasAgg(fig2)
agg_canvas.draw()
agg_renderer = agg_canvas.get_renderer()
agg_results = {
    "title_via_agg": ax2.title.get_window_extent(renderer=agg_renderer).width > 1,
    "xlabel_via_agg": ax2.xaxis.label.get_window_extent(renderer=agg_renderer).width > 1,
    "ylabel_via_agg": ax2.yaxis.label.get_window_extent(renderer=agg_renderer).width > 1,
}
print("\nPart 2: Agg renderer")
for name, ok in agg_results.items():
    print(f"  {name:<16}: {'OK' if ok else 'FAILED'}")
plt.close("all")

if all(results.values()) and all(agg_results.values()):
    print("\nALL CHECKS PASS, including the Agg/savefig-specific path.")
else:
    print("\nAT LEAST ONE CHECK FAILED: if Part 2 failed while Part 1 passed, that is direct evidence of a "
          "live-backend-vs-Agg discrepancy on this machine, which is exactly the kind of thing that would "
          "explain a blank saved PNG with no other symptom. Worth reporting back which backend Section 0 "
          "printed above, plus this result.")


## Section 0.5: Campaign diagnostics and static p95 episodes


In [ ]:
def static_episodes_with_end(above_mask, persist_steps=2, merge_gap=pd.Timedelta(hours=1)):
    arr = above_mask.fillna(False).values.astype(int)
    idx = above_mask.index
    diff = np.diff(np.concatenate([[0], arr, [0]]))
    starts = np.where(diff==1)[0]; ends = np.where(diff==-1)[0]-1
    episodes = [(idx[s], idx[e]) for s,e in zip(starts,ends) if (e-s+1) >= persist_steps]
    if not episodes: return pd.DataFrame(columns=["alarm_time","alarm_end"])
    merged = [episodes[0]]
    for s,e in episodes[1:]:
        if s - merged[-1][1] <= merge_gap: merged[-1] = (merged[-1][0], e)
        else: merged.append((s,e))
    return pd.DataFrame({"alarm_time":[m[0] for m in merged], "alarm_end":[m[1] for m in merged]})

def dominant_condition(c, activity, pad_h=0):
    if "hydrologic_condition" not in activity.columns: return None
    pad = pd.Timedelta(hours=pad_h)
    sub = activity[(activity["activity_datetime"]>=c["first_sample_time"]-pad) &
                     (activity["activity_datetime"]<=c["last_sample_time"]+pad)]
    if len(sub)==0: return None
    vc = sub["hydrologic_condition"].value_counts()
    return vc.index[0] if len(vc) else None

PAD_H = 6
STATION_CAMPAIGN_DIAG = {}
STATION_STATIC_EPISODES = {}
STATION_DIS_GRID = {}
STATION_TUR_GRID = {}

for site_id in ALL_STATION_IDS:
    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"]).set_index("datetime")
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"])
    activity = pd.read_csv(PROCESSED_DIR / f"{site_id}_activity_clean.csv", parse_dates=["activity_datetime"])
    eligibility = pd.read_csv(PROCESSED_DIR / f"{site_id}_eligibility_audit.csv",
                                parse_dates=["first_sample_time","last_sample_time"])
    dis, tur = iv["discharge_cfs"], iv["turbidity_fnu"]
    dis_valid = dis.dropna()
    STATION_DIS_GRID[site_id] = dis
    STATION_TUR_GRID[site_id] = tur

    rows = []
    for _, c in campaigns.iterrows():
        lo, hi = c["first_sample_time"]-pd.Timedelta(hours=PAD_H), c["last_sample_time"]+pd.Timedelta(hours=PAD_H)
        win = dis.loc[lo:hi]
        peak = win.max() if win.notna().any() else np.nan
        pctile = float((dis_valid < peak).mean()*100) if pd.notna(peak) else np.nan
        duration_h = (c["last_sample_time"]-c["first_sample_time"]).total_seconds()/3600
        pre_win = dis.loc[c["first_sample_time"]-pd.Timedelta(hours=PAD_H):c["first_sample_time"]].dropna()
        rise_rate = ((pre_win.iloc[-1]-pre_win.iloc[0])/PAD_H) if len(pre_win)>=2 else np.nan
        variability = float(win.std()) if win.notna().sum()>1 else np.nan
        rows.append(dict(campaign_id=c["campaign_id"], peak_discharge_cfs=peak, discharge_percentile=pctile,
                           duration_h=round(duration_h,2), pre_sample_rise_rate_cfs_per_h=rise_rate,
                           discharge_variability=variability, dominant_hydrologic_condition=dominant_condition(c, activity)))
    diag = pd.DataFrame(rows).merge(
        eligibility[["campaign_id","discharge_coverage_frac","turbidity_coverage_frac"]], on="campaign_id", how="left")
    STATION_CAMPAIGN_DIAG[site_id] = diag

    q95 = float(np.nanpercentile(dis_valid, 95)) if len(dis_valid) else np.nan
    static_ep = static_episodes_with_end(dis > q95) if pd.notna(q95) else pd.DataFrame(columns=["alarm_time","alarm_end"])
    static_ep["duration_h"] = (static_ep["alarm_end"]-static_ep["alarm_time"]).dt.total_seconds()/3600 if len(static_ep) else []
    STATION_STATIC_EPISODES[site_id] = static_ep

    print(f"{site_id}: {len(diag)} campaign diagnostics, {len(static_ep)} static p95 episodes "
          f"(median duration {static_ep['duration_h'].median():.2f}h)" if len(static_ep) else f"{site_id}: {len(diag)} campaign diagnostics, 0 static episodes")

    if SAVE_OUTPUTS:
        save_csv(diag, PROCESSED_DIR / f"{site_id}_campaign_diagnostics.csv", f"{site_id} campaign diagnostics")
        save_csv(static_ep, PROCESSED_DIR / f"{site_id}_static_p95_episodes_full.csv", f"{site_id} static episodes (full record)")


## Group A: Data Availability and Station Comparability


### Figure 1: Study-Site and Data-Summary Overview


In [ ]:
station_meta = pd.read_csv(RAW_DIR / "station.csv", low_memory=False)
station_meta = station_meta[station_meta["MonitoringLocationIdentifier"].str.startswith("USGS-", na=False)].copy()
station_meta["site_id"] = station_meta["MonitoringLocationIdentifier"].str.replace("USGS-", "", regex=False)

summary = pd.read_csv(TABLE_DIR / "makiki_05a_cross_station_summary.csv", dtype={"site_id": str})

fig01_rows = []
for site_id in ALL_STATION_IDS:
    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"])
    meta_row = station_meta[station_meta["site_id"]==site_id]
    summ_row = summary[summary["site_id"]==site_id].iloc[0]
    fig01_rows.append(dict(
        site_id=site_id, name=STATION_NAMES[site_id],
        drainage_area_sqmi=float(meta_row["DrainageAreaMeasure/MeasureValue"].iloc[0]) if len(meta_row) else None,
        latitude=float(meta_row["LatitudeMeasure"].iloc[0]) if len(meta_row) else None,
        longitude=float(meta_row["LongitudeMeasure"].iloc[0]) if len(meta_row) else None,
        continuous_data_start=str(iv["datetime"].min()), continuous_data_end=str(iv["datetime"].max()),
        turbidity_record_start=summ_row["turbidity_record_start"],
        n_campaigns=int(summ_row["n_campaigns"]),
        n_discharge_eligible=int(summ_row["n_discharge_eligible"]),
        n_turbidity_eligible=int(summ_row["n_turbidity_eligible"]),
        n_joint_eligible=int(summ_row["n_joint_eligible"]),
    ))
FIG01_TABLE = pd.DataFrame(fig01_rows)
print("Figure 1: station summary table:")
print(FIG01_TABLE.to_string(index=False))
if SAVE_OUTPUTS:
    save_csv(FIG01_TABLE, FIG_DIRS["A"] / "fig01_station_summary.csv")

if FIG01_TABLE["latitude"].notna().all():
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(FIG01_TABLE["longitude"], FIG01_TABLE["latitude"], s=120, c=["#888888" if s==MAKIKI_ID else "#1a6faf" for s in FIG01_TABLE["site_id"]])
    for _, r in FIG01_TABLE.iterrows():
        ax.annotate(f"{r['site_id']}\n({r['name'].split(chr(32))[0]})", (r["longitude"], r["latitude"]),
                     textcoords="offset points", xytext=(8,5), fontsize=8)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("Station locations (Oahu): compact map, no basemap tiles")
    ax.set_aspect("equal")
    save_fig("A", "fig01_station_map.png", fig)
else:
    print("Coordinates missing for at least one station: map panel skipped, table above still complete.")


### Figure 2: Continuous-Data Availability Timeline


In [ ]:
PARAM_PALETTE = {"00095":"#1f9e89","00631":"#e76f51","00665":"#d62828","00530":"#457b9d",
                   "80154":"#7b2cbf","80155":"#f4a261"}
extra_palette_pool = plt.cm.tab20.colors

for site_id in ALL_STATION_IDS:
    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"])
    results_path = find_file([f"{site_id}_results.csv"], dir_=RAW_DIR, required=False)

    fig, axes = plt.subplots(3, 1, figsize=(14,4), sharex=True, gridspec_kw=dict(height_ratios=[1,1,1.3]))

    dis_avail = iv["discharge_cfs"].notna().astype(int)
    axes[0].fill_between(iv["datetime"], 0, dis_avail, step="mid", color="#a8d5e2")
    axes[0].set_yticks([]); axes[0].set_ylabel("Discharge\nIV", fontsize=8)

    tur_avail = iv["turbidity_fnu"].notna().astype(int)
    axes[1].fill_between(iv["datetime"], 0, tur_avail, step="mid", color="#f4c89a")
    axes[1].set_yticks([]); axes[1].set_ylabel("Turbidity\nIV", fontsize=8)

    if results_path is not None:
        chem = pd.read_csv(results_path, low_memory=False)
        chem["ActivityStartDate"] = pd.to_datetime(chem["ActivityStartDate"])
        codes_present = chem["USGSPCode"].astype(str).value_counts().head(8).index.tolist()
        used_palette = {}
        for i, code in enumerate(codes_present):
            used_palette[code] = PARAM_PALETTE.get(code, extra_palette_pool[i % len(extra_palette_pool)])
        for code in codes_present:
            sub = chem[chem["USGSPCode"].astype(str)==code]
            axes[2].vlines(sub["ActivityStartDate"], 0, 1, color=used_palette[code], lw=1.0, label=f"{code}")
        axes[2].legend(fontsize=6.5, ncol=4, loc="upper center", bbox_to_anchor=(0.5,-0.35))
    else:
        activity = pd.read_csv(PROCESSED_DIR / f"{site_id}_activity_clean.csv", parse_dates=["activity_datetime"])
        axes[2].vlines(activity["activity_datetime"], 0, 1, color="#888888", lw=1.0)
        print(f"  {site_id}: no raw results.csv found: grab samples shown ungrouped (no chemistry-parameter color).")
    axes[2].set_yticks([]); axes[2].set_ylabel("WQP grab\nsamples", fontsize=8)

    if site_id == MAKIKI_ID:
        for ax in axes:
            ax.axvline(pd.Timestamp("2021-12-18", tz="UTC"), color="black", ls=":", lw=1.2, alpha=0.7)
        axes[0].text(pd.Timestamp("2021-12-18", tz="UTC"), 1.05, "dev/test split\n(Makiki only)", fontsize=6.5, ha="center")

    fig.suptitle(f"{site_id} ({STATION_NAMES[site_id]}): data availability", fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_fig("A", f"fig02_availability_timeline_{site_id}.png", fig)


### Figure 3: Coverage and Eligibility by Station


In [ ]:
summary = pd.read_csv(TABLE_DIR / "makiki_05a_cross_station_summary.csv", dtype={"site_id": str})
summary = summary.set_index("site_id").loc[ALL_STATION_IDS].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13,5))
x = np.arange(len(ALL_STATION_IDS)); width = 0.35
axes[0].bar(x-width/2, summary["discharge_coverage"]*100, width, label="discharge", color=COLOR_DISCHARGE_BOCPD)
axes[0].bar(x+width/2, summary["turbidity_coverage"]*100, width, label="turbidity", color=COLOR_TURBIDITY_BOCPD)
axes[0].set_xticks(x); axes[0].set_xticklabels(ALL_STATION_IDS, fontsize=8.5)
axes[0].set_ylabel("Coverage (%)"); axes[0].legend(fontsize=8.5); axes[0].set_title("Sensor coverage")

w3 = 0.25
axes[1].bar(x-w3, summary["n_discharge_eligible"], w3, label="discharge-eligible", color=COLOR_DISCHARGE_BOCPD)
axes[1].bar(x, summary["n_turbidity_eligible"], w3, label="turbidity-eligible", color=COLOR_TURBIDITY_BOCPD)
axes[1].bar(x+w3, summary["n_joint_eligible"], w3, label="joint-eligible", color="#7b2cbf")
axes[1].set_xticks(x); axes[1].set_xticklabels(ALL_STATION_IDS, fontsize=8.5)
axes[1].set_ylabel("Eligible campaigns"); axes[1].legend(fontsize=8.5); axes[1].set_title("Eligibility counts")
plt.tight_layout()
save_fig("A", "fig03_coverage_eligibility.png", fig)


### Figure 4: Cadence and Missingness Diagnostics


In [ ]:
def gap_diagnostics(iv, col):
    s = iv[col]
    valid = s.notna().astype(int).values
    diff = np.diff(np.concatenate([[0], valid, [0]]))
    starts = np.where(diff==1)[0]; ends = np.where(diff==-1)[0]-1
    seg_lens = (ends - starts + 1)
    missing = 100*(1 - s.notna().mean())
    gap_mask = (~s.notna()).astype(int).values
    gdiff = np.diff(np.concatenate([[0], gap_mask, [0]]))
    gstarts = np.where(gdiff==1)[0]; gends = np.where(gdiff==-1)[0]-1
    n_gaps = len(gstarts)
    return dict(missingness_pct=round(missing,2), n_gaps=n_gaps,
                 longest_segment_steps=int(seg_lens.max()) if len(seg_lens) else 0)

summary = pd.read_csv(TABLE_DIR / "makiki_05a_cross_station_summary.csv", dtype={"site_id": str}).set_index("site_id")
fig04_rows = []
for site_id in ALL_STATION_IDS:
    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"]).set_index("datetime")
    cadence_min = summary.loc[site_id, "cadence_sec"]/60
    dgap = gap_diagnostics(iv, "discharge_cfs"); tgap = gap_diagnostics(iv, "turbidity_fnu")
    fig04_rows.append(dict(site_id=site_id, cadence_min=cadence_min,
                             cadence_majority_frac=summary.loc[site_id,"cadence_majority_frac"],
                             discharge_missingness_pct=dgap["missingness_pct"], discharge_n_gaps=dgap["n_gaps"],
                             discharge_longest_segment_days=round(dgap["longest_segment_steps"]*cadence_min/1440,1),
                             turbidity_missingness_pct=tgap["missingness_pct"], turbidity_n_gaps=tgap["n_gaps"],
                             turbidity_longest_segment_days=round(tgap["longest_segment_steps"]*cadence_min/1440,1)))
FIG04_TABLE = pd.DataFrame(fig04_rows)
print(FIG04_TABLE.to_string(index=False))
if SAVE_OUTPUTS:
    save_csv(FIG04_TABLE, FIG_DIRS["A"] / "fig04_cadence_missingness.csv")

fig, axes = plt.subplots(1, 2, figsize=(12,5))
x = np.arange(len(ALL_STATION_IDS)); w = 0.35
axes[0].bar(x-w/2, FIG04_TABLE["discharge_missingness_pct"], w, label="discharge", color=COLOR_DISCHARGE_BOCPD)
axes[0].bar(x+w/2, FIG04_TABLE["turbidity_missingness_pct"], w, label="turbidity", color=COLOR_TURBIDITY_BOCPD)
axes[0].set_xticks(x); axes[0].set_xticklabels(ALL_STATION_IDS, fontsize=8.5)
axes[0].set_ylabel("Missingness (%)"); axes[0].legend(fontsize=8.5); axes[0].set_title("Missingness")

axes[1].bar(x-w/2, FIG04_TABLE["discharge_longest_segment_days"], w, label="discharge", color=COLOR_DISCHARGE_BOCPD)
axes[1].bar(x+w/2, FIG04_TABLE["turbidity_longest_segment_days"], w, label="turbidity", color=COLOR_TURBIDITY_BOCPD)
axes[1].set_xticks(x); axes[1].set_xticklabels(ALL_STATION_IDS, fontsize=8.5)
axes[1].set_ylabel("Longest continuous segment (days)"); axes[1].legend(fontsize=8.5); axes[1].set_title("Longest segment")
plt.tight_layout()
save_fig("A", "fig04_cadence_missingness.png", fig)


### Figure 5: Campaign Distribution Through Time


In [ ]:
condition_colors = {"Rising stage":"#e07b39", "Falling stage":"#457b9d", "Peak stage":"#d62828",
                      "Stable, normal stage":"#888888", None: "#cccccc"}

fig, axes = plt.subplots(len(ALL_STATION_IDS), 1, figsize=(13, 1.6*len(ALL_STATION_IDS)), sharex=True)
for i, site_id in enumerate(ALL_STATION_IDS):
    ax = axes[i]
    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"]).set_index("datetime")
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"])
    diag = STATION_CAMPAIGN_DIAG[site_id]
    merged = campaigns.merge(diag[["campaign_id","dominant_hydrologic_condition"]], on="campaign_id", how="left")

    dis_avail = iv["discharge_cfs"].notna()
    ax.fill_between(iv.index, 0, 1, where=dis_avail.values, color="#dceefb", step="mid", alpha=0.6)
    for _, c in merged.iterrows():
        color = condition_colors.get(c["dominant_hydrologic_condition"], "#cccccc")
        ax.axvline(c["first_sample_time"], color=color, lw=1.0, ymin=0.1, ymax=0.9)
    ax.set_yticks([]); ax.set_ylabel(site_id, fontsize=8.5, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("Date")
handles = [plt.Line2D([0],[0], color=c, lw=2, label=k or "unknown") for k,c in condition_colors.items() if k]
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02), fontsize=8.5)
fig.suptitle("Campaign timing by station, colored by hydrologic condition", y=1.06, fontsize=11, fontweight="bold")
plt.tight_layout()
save_fig("A", "fig05_campaign_raster.png", fig)


## Group B: Primary Cross-Station Performance

Loads the final 05b results and saved alarm and match tables.


In [ ]:
FINAL = pd.read_csv(TABLE_DIR / "makiki_05b_cross_station_final.csv", dtype={"site_id": str})
STATION_ORDER = [MAKIKI_ID] + list(EXTERNAL_STATIONS.keys())
STATION_LABELS = {MAKIKI_ID: "Makiki\n(reference)", **{k: k for k in EXTERNAL_STATIONS}}

ALL_MATCHES = {}
ALL_ALARMS = {}
for site_id in ALL_STATION_IDS:
    ALL_MATCHES[site_id] = {}
    ALL_ALARMS[site_id] = {}
    for method in ["discharge_bocpd", "static_discharge_p95", "turbidity_bocpd"]:
        mp = PROCESSED_DIR / f"{site_id}_matches_{method}.csv"
        ap = PROCESSED_DIR / f"{site_id}_alarms_{method}.csv"
        if mp.exists(): ALL_MATCHES[site_id][method] = pd.read_csv(mp)
        if ap.exists(): ALL_ALARMS[site_id][method] = pd.read_csv(ap, parse_dates=["alarm_time"])
print("Loaded final results table and per-station match/alarm files.")
print(f"FINAL table: {len(FINAL)} rows")


### Figure 6: Recall by Station and Method


In [ ]:
fig, ax = plt.subplots(figsize=(10,5.5))
x_base = np.arange(len(STATION_ORDER))
offsets = {"discharge_bocpd": -0.15, "static_discharge_p95": 0.0, "turbidity_bocpd": 0.15}
for method in ["discharge_bocpd","static_discharge_p95","turbidity_bocpd"]:
    sub = FINAL[FINAL["method"]==method].set_index("site_id")
    xs, ys, yerr_lo, yerr_hi = [], [], [], []
    for i, site_id in enumerate(STATION_ORDER):
        if site_id not in sub.index: continue
        r = sub.loc[site_id]
        xs.append(i + offsets[method]); ys.append(r["recall"])
        yerr_lo.append(r["recall"]-r["ci_lo"]); yerr_hi.append(r["ci_hi"]-r["recall"])
    ax.errorbar(xs, ys, yerr=[yerr_lo,yerr_hi], fmt=METHOD_MARKERS[method]+"-", color=METHOD_COLORS[method],
                 capsize=4, label=method, alpha=0.5 if method=="turbidity_bocpd" else 1.0)
ax.axvline(0.5, color="grey", ls=":", lw=1, alpha=0.6)
ax.set_xticks(x_base); ax.set_xticklabels([STATION_LABELS[s] for s in STATION_ORDER])
ax.set_ylabel("Campaign recall (95% Wilson CI)"); ax.set_ylim(-0.05,1.08)
ax.legend(fontsize=8.5, loc="lower left")
ax.set_title("Recall by station and method (turbidity shown lighter: secondary)")
plt.tight_layout()
save_fig("B", "fig06_recall_by_station.png", fig)


### Figure 7: Alarm Burden by Station and Method


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5.5))
for method in ["discharge_bocpd","static_discharge_p95"]:
    sub = FINAL[FINAL["method"]==method].set_index("site_id").reindex(STATION_ORDER)
    axes[0].plot(range(len(STATION_ORDER)), sub["unmatched_alarms_per_month"], METHOD_MARKERS[method]+"-",
                  color=METHOD_COLORS[method], label=method)
axes[0].set_xticks(range(len(STATION_ORDER))); axes[0].set_xticklabels([STATION_LABELS[s] for s in STATION_ORDER])
axes[0].set_ylabel("Unmatched alarms / month"); axes[0].legend(fontsize=8.5); axes[0].set_title("Primary methods")

tur_sub = FINAL[FINAL["method"]=="turbidity_bocpd"].set_index("site_id").reindex(STATION_ORDER)
axes[1].bar(range(len(STATION_ORDER)), tur_sub["unmatched_alarms_per_month"], color=COLOR_TURBIDITY_BOCPD, alpha=0.7)
axes[1].set_xticks(range(len(STATION_ORDER))); axes[1].set_xticklabels([STATION_LABELS[s] for s in STATION_ORDER])
axes[1].set_ylabel("Unmatched alarms / month"); axes[1].set_title("Turbidity (secondary): separate panel, different scale")
plt.tight_layout()
save_fig("B", "fig07_alarm_burden.png", fig)


### Figure 8: Recall versus alarm burden


In [ ]:
fig, ax = plt.subplots(figsize=(8.5,6.5))
for method in ["discharge_bocpd","static_discharge_p95","turbidity_bocpd"]:
    sub = FINAL[FINAL["method"]==method]
    for _, r in sub.iterrows():
        is_makiki = (r["site_id"]==MAKIKI_ID)
        ax.scatter(r["unmatched_alarms_per_month"], r["recall"], marker=METHOD_MARKERS[method],
                    s=160 if is_makiki else 100, facecolor=METHOD_COLORS[method],
                    edgecolor="black" if is_makiki else "none", linewidth=1.5 if is_makiki else 0,
                    alpha=0.5 if method=="turbidity_bocpd" else 0.9)
        ax.annotate(r["site_id"] if not is_makiki else "Makiki (ref.)", (r["unmatched_alarms_per_month"], r["recall"]),
                     fontsize=7, textcoords="offset points", xytext=(5,5))
handles = [plt.Line2D([0],[0], marker=METHOD_MARKERS[m], color=METHOD_COLORS[m], lw=0, ms=9, label=m)
           for m in ["discharge_bocpd","static_discharge_p95","turbidity_bocpd"]]
handles.append(plt.Line2D([0],[0], marker="o", color="white", markeredgecolor="black", markeredgewidth=1.5, lw=0, ms=9, label="Makiki (development-site reference)"))
ax.legend(handles=handles, fontsize=8, loc="lower right")
ax.set_xlabel("Unmatched alarms / month"); ax.set_ylabel("Campaign recall")
ax.set_title("Recall vs alarm burden: headline operational figure")
plt.tight_layout()
save_fig("B", "fig08_recall_vs_burden.png", fig)


### Figure 9: Matched-Alarm Fraction by Station


In [ ]:
fig, ax = plt.subplots(figsize=(11,5.5))
methods = ["discharge_bocpd","static_discharge_p95","turbidity_bocpd"]
n_methods = len(methods)
bar_w = 0.8/n_methods
x_base = np.arange(len(STATION_ORDER))
for mi, method in enumerate(methods):
    sub = FINAL[FINAL["method"]==method].set_index("site_id").reindex(STATION_ORDER)
    xs = x_base + (mi-1)*bar_w
    total_alarms = []
    for site_id in STATION_ORDER:
        a = ALL_ALARMS.get(site_id, {}).get(method)
        total_alarms.append(len(a) if a is not None else np.nan)
    ax.bar(xs, total_alarms, bar_w, color=METHOD_COLORS[method], alpha=0.35, label=f"{method} (all alarms)" if mi==0 else None)
    matched = sub["matched_alarm_fraction"] * pd.Series(total_alarms, index=STATION_ORDER)
    ax.bar(xs, matched.values, bar_w, color=METHOD_COLORS[method], alpha=0.95)
handles = [plt.Rectangle((0,0),1,1, color=METHOD_COLORS[m], alpha=0.95, label=f"{m} (matched)") for m in methods]
handles += [plt.Rectangle((0,0),1,1, color="grey", alpha=0.35, label="all alarms (unmatched portion)")]
ax.legend(handles=handles, fontsize=7.5, loc="upper right")
ax.set_xticks(x_base); ax.set_xticklabels([STATION_LABELS[s] for s in STATION_ORDER])
ax.set_ylabel("Total record-period alarm count")
ax.set_title("All alarms vs matched alarms, by station and method")
plt.tight_layout()
save_fig("B", "fig09_matched_fraction.png", fig)


### Figure 10: Duplicate Alarms per Detected Campaign


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5.5))
for method in ["discharge_bocpd","static_discharge_p95"]:
    sub = FINAL[FINAL["method"]==method].set_index("site_id").reindex(STATION_ORDER)
    axes[0].plot(range(len(STATION_ORDER)), sub["duplicate_alarms_per_campaign"], METHOD_MARKERS[method]+"-",
                  color=METHOD_COLORS[method], label=method)
axes[0].axhline(1.0, color="grey", ls=":", lw=1, label="1 alarm/campaign (ideal)")
axes[0].set_xticks(range(len(STATION_ORDER))); axes[0].set_xticklabels([STATION_LABELS[s] for s in STATION_ORDER])
axes[0].set_ylabel("Mean duplicate alarms / detected campaign"); axes[0].legend(fontsize=8); axes[0].set_title("Mean, primary methods")

dist_rows = []
for site_id in EXTERNAL_STATIONS:
    for method in ["discharge_bocpd","static_discharge_p95"]:
        m = ALL_MATCHES.get(site_id, {}).get(method)
        if m is not None:
            det = m[m["detected"]==True]
            for v in det["n_alarms_in_window"]:
                dist_rows.append(dict(site_id=site_id, method=method, n_alarms_in_window=v))
dist_df = pd.DataFrame(dist_rows)
if len(dist_df):
    positions, labels_, colors_ = [], [], []
    pos = 0
    for site_id in EXTERNAL_STATIONS:
        for method in ["discharge_bocpd","static_discharge_p95"]:
            vals = dist_df[(dist_df["site_id"]==site_id)&(dist_df["method"]==method)]["n_alarms_in_window"]
            if len(vals):
                axes[1].boxplot(vals, positions=[pos], widths=0.6,
                                  patch_artist=True, boxprops=dict(facecolor=METHOD_COLORS[method], alpha=0.6))
                positions.append(pos); labels_.append(f"{site_id}\n{method.split(chr(95))[0]}"); pos += 1
        pos += 0.5
    axes[1].set_xticks(positions); axes[1].set_xticklabels(labels_, fontsize=6.5)
    axes[1].set_ylabel("Alarms in matching window (per detected campaign)")
    axes[1].set_title("Distribution, external stations")
plt.tight_layout()
save_fig("B", "fig10_duplicate_alarms.png", fig)


### Figure 11: Lead/Lag Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
for method in ["discharge_bocpd","static_discharge_p95"]:
    all_lags = []
    for site_id in EXTERNAL_STATIONS:
        m = ALL_MATCHES.get(site_id, {}).get(method)
        if m is not None:
            all_lags.extend(m.loc[m["detected"]==True, "lead_lag_minutes"].dropna().tolist())
    if all_lags:
        sorted_lags = np.sort(all_lags)
        ecdf = np.arange(1, len(sorted_lags)+1) / len(sorted_lags)
        ax.step(sorted_lags, ecdf, where="post", color=METHOD_COLORS[method], lw=2, label=f"{method} (n={len(all_lags)})")
ax.axvline(0, color="black", lw=1, alpha=0.5)
ax.set_xlabel("Lead/lag (minutes): negative = detector led field sample")
ax.set_ylabel("ECDF, matched campaigns (external stations pooled)")
ax.legend(fontsize=9)
ax.set_title("Lead/lag distribution, external stations pooled")
plt.tight_layout()
save_fig("B", "fig11_lead_lag_ecdf.png", fig)


## Group C: Campaign-Level Transfer Behavior

Discharge BOCPD and static p95 use the same eligible campaigns. Turbidity is separate because its eligible set differs.


### Figure 12: Campaign Detection Heatmap


In [ ]:
fig, axes = plt.subplots(len(EXTERNAL_STATIONS), 1, figsize=(10, 2.2*len(EXTERNAL_STATIONS)))
for i, site_id in enumerate(EXTERNAL_STATIONS):
    ax = axes[i] if len(EXTERNAL_STATIONS) > 1 else axes
    methods = ["discharge_bocpd", "static_discharge_p95", "turbidity_bocpd"]
    all_campaign_ids = sorted(set().union(*[set(ALL_MATCHES[site_id][m]["campaign_id"]) for m in methods if m in ALL_MATCHES[site_id]]))
    grid = np.full((len(all_campaign_ids), len(methods)), np.nan)
    for mi, method in enumerate(methods):
        if method not in ALL_MATCHES[site_id]: continue
        m = ALL_MATCHES[site_id][method].set_index("campaign_id")
        for ci, cid in enumerate(all_campaign_ids):
            if cid in m.index:
                grid[ci, mi] = 1.0 if m.loc[cid,"detected"] else 0.0
    im = ax.imshow(grid.T, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, interpolation="none")
    ax.set_yticks(range(len(methods))); ax.set_yticklabels(methods, fontsize=7.5)
    ax.set_xticks([]); ax.set_xlabel(f"campaigns (n={len(all_campaign_ids)})", fontsize=8)
    ax.set_title(f"{site_id}", fontsize=9, loc="left")
fig.suptitle("Campaign detection heatmap (green=detected, red=missed, white=not eligible for that method)", fontsize=10, y=1.02)
plt.tight_layout()
save_fig("C", "fig12_detection_heatmap.png", fig)


### Figures 13 & 17: Method Agreement (Matrix and Stacked Bars)


In [ ]:
agreement_rows = []
for site_id in EXTERNAL_STATIONS:
    m_b = ALL_MATCHES[site_id].get("discharge_bocpd")
    m_s = ALL_MATCHES[site_id].get("static_discharge_p95")
    if m_b is None or m_s is None: continue
    merged = m_b[["campaign_id","detected"]].rename(columns={"detected":"bocpd"}).merge(
        m_s[["campaign_id","detected"]].rename(columns={"detected":"static"}), on="campaign_id")
    both = int((merged["bocpd"] & merged["static"]).sum())
    bocpd_only = int((merged["bocpd"] & ~merged["static"]).sum())
    static_only = int((~merged["bocpd"] & merged["static"]).sum())
    neither = int((~merged["bocpd"] & ~merged["static"]).sum())
    agreement_rows.append(dict(site_id=site_id, both=both, bocpd_only=bocpd_only, static_only=static_only, neither=neither))

AGREEMENT = pd.DataFrame(agreement_rows)
print(AGREEMENT.to_string(index=False))
if SAVE_OUTPUTS:
    save_csv(AGREEMENT, FIG_DIRS["C"] / "fig13_17_method_agreement.csv")

fig, ax = plt.subplots(figsize=(8.5,5.5))
x = np.arange(len(AGREEMENT))
bottom = np.zeros(len(AGREEMENT))
labels_cats = ["both","bocpd_only","static_only","neither"]
colors_cats = ["#2a9d8f","#1a6faf","#e07b39","#c1121f"]
for cat, col in zip(labels_cats, colors_cats):
    ax.bar(x, AGREEMENT[cat], bottom=bottom, color=col, label=cat)
    bottom += AGREEMENT[cat].values
ax.set_xticks(x); ax.set_xticklabels(AGREEMENT["site_id"])
ax.set_ylabel("Campaign count"); ax.legend(fontsize=8.5)
ax.set_title("Method agreement: discharge BOCPD vs static p95 (stacked)")
plt.tight_layout()
save_fig("C", "fig13_17_method_agreement.png", fig)


### Figure 14: Campaign-Level Recall Difference (Paired Bootstrap CI)


In [ ]:
N_BOOT = 2000
delta_rows = []
for site_id in EXTERNAL_STATIONS:
    m_b = ALL_MATCHES[site_id].get("discharge_bocpd")
    m_s = ALL_MATCHES[site_id].get("static_discharge_p95")
    if m_b is None or m_s is None: continue
    merged = m_b[["campaign_id","detected"]].rename(columns={"detected":"bocpd"}).merge(
        m_s[["campaign_id","detected"]].rename(columns={"detected":"static"}), on="campaign_id")
    n = len(merged)
    obs_delta = merged["bocpd"].mean() - merged["static"].mean()
    idx_arr = np.arange(n)
    deltas = np.array([merged.iloc[RNG.choice(idx_arr, size=n, replace=True)].pipe(
        lambda s: s["bocpd"].mean()-s["static"].mean()) for _ in range(N_BOOT)])
    ci_lo, ci_hi = np.percentile(deltas, [2.5, 97.5])
    delta_rows.append(dict(site_id=site_id, delta_recall=round(obs_delta,4), boot_ci_lo=round(ci_lo,4), boot_ci_hi=round(ci_hi,4)))

DELTA_RECALL = pd.DataFrame(delta_rows)
print(DELTA_RECALL.to_string(index=False))
if SAVE_OUTPUTS:
    save_csv(DELTA_RECALL, FIG_DIRS["C"] / "fig14_recall_difference.csv")

fig, ax = plt.subplots(figsize=(7,5))
x = np.arange(len(DELTA_RECALL))
ax.errorbar(x, DELTA_RECALL["delta_recall"],
             yerr=[DELTA_RECALL["delta_recall"]-DELTA_RECALL["boot_ci_lo"], DELTA_RECALL["boot_ci_hi"]-DELTA_RECALL["delta_recall"]],
             fmt="o", color="#1a6faf", capsize=5, ms=9)
ax.axhline(0, color="black", lw=1)
ax.set_xticks(x); ax.set_xticklabels(DELTA_RECALL["site_id"])
ax.set_ylabel("Recall(discharge BOCPD) - Recall(static p95)")
ax.set_title("Campaign-level recall difference, paired bootstrap 95% CI")
plt.tight_layout()
save_fig("C", "fig14_recall_difference.png", fig)


### Figure 15: Campaign Difficulty Characteristics


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,7))
chars = ["peak_discharge_cfs","duration_h","pre_sample_rise_rate_cfs_per_h","turbidity_coverage_frac",
          "discharge_variability","discharge_coverage_frac"]
for row, method in enumerate(["discharge_bocpd","static_discharge_p95"]):
    detected_vals = {c: [] for c in chars}; missed_vals = {c: [] for c in chars}
    for site_id in EXTERNAL_STATIONS:
        m = ALL_MATCHES[site_id].get(method)
        diag = STATION_CAMPAIGN_DIAG[site_id]
        if m is None: continue
        merged = m.merge(diag, on="campaign_id", how="left")
        for c in chars:
            detected_vals[c].extend(merged.loc[merged["detected"]==True, c].dropna().tolist())
            missed_vals[c].extend(merged.loc[merged["detected"]==False, c].dropna().tolist())
    for ci, c in enumerate(chars[:4]):
        ax = axes[row, ci]
        data = [detected_vals[c], missed_vals[c]]
        if any(len(d) for d in data):
            ax.boxplot(data, tick_labels=["detected","missed"], widths=0.5)
        ax.set_title(c, fontsize=7.5)
        if ci==0: ax.set_ylabel(method, fontsize=8.5)
fig.suptitle("Campaign difficulty characteristics: detected vs missed (external stations pooled)", fontsize=11, y=1.0)
plt.tight_layout()
save_fig("C", "fig15_campaign_difficulty.png", fig)


### Figure 16: Campaign-Level Lead/Lag Paired Comparison


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
for site_id in EXTERNAL_STATIONS:
    m_b = ALL_MATCHES[site_id].get("discharge_bocpd")
    m_s = ALL_MATCHES[site_id].get("static_discharge_p95")
    if m_b is None or m_s is None: continue
    merged = m_b[["campaign_id","detected","lead_lag_minutes"]].rename(columns={"detected":"det_b","lead_lag_minutes":"lag_b"}).merge(
        m_s[["campaign_id","detected","lead_lag_minutes"]].rename(columns={"detected":"det_s","lead_lag_minutes":"lag_s"}), on="campaign_id")
    both = merged[merged["det_b"] & merged["det_s"]]
    for _, r in both.iterrows():
        ax.plot([0,1], [r["lag_b"], r["lag_s"]], color="grey", alpha=0.4, lw=1)
    ax.scatter([0]*len(both), both["lag_b"], color=COLOR_DISCHARGE_BOCPD, zorder=3, s=30)
    ax.scatter([1]*len(both), both["lag_s"], color=COLOR_STATIC_P95, zorder=3, s=30)
ax.set_xticks([0,1]); ax.set_xticklabels(["discharge_bocpd","static_discharge_p95"])
ax.axhline(0, color="black", lw=0.8, alpha=0.5)
ax.set_ylabel("Lead/lag (minutes)")
ax.set_title("Paired lead/lag, campaigns detected by both methods (external stations pooled)")
plt.tight_layout()
save_fig("C", "fig16_paired_lead_lag.png", fig)


## Group D: BOCPD Mechanism and Score Behavior

Uses the score fields saved by 05b. Expected run-length drop is unavailable and is not substituted.


In [ ]:
STATION_INSTR = {}
for site_id in ALL_STATION_IDS:
    STATION_INSTR[site_id] = {}
    for var in ["discharge","turbidity"]:
        p = PROCESSED_DIR / f"{site_id}_bocpd_instrumented_{var}.csv.gz"
        if p.exists():
            df = pd.read_csv(p, parse_dates=["datetime"])
            STATION_INSTR[site_id][var] = df
        else:
            print(f"  {site_id}/{var}: instrumented scores not found: figures needing this will be skipped for this station/variable.")

def near_away_mask(instr_df, campaigns, pad_h=6):
    dts = pd.DatetimeIndex(instr_df["datetime"])
    near = np.zeros(len(dts), dtype=bool)
    pad = pd.Timedelta(hours=pad_h)
    for _, c in campaigns.iterrows():
        near |= (dts >= c["first_sample_time"]-pad) & (dts <= c["last_sample_time"]+pad)
    return near

print("Loaded instrumented scores for all stations where available.")


### Figure 18: Representative Time-Series Case Studies

Shows up to four detection outcomes per station, omitting outcomes with no campaigns.


In [ ]:
for site_id in EXTERNAL_STATIONS:
    m_b = ALL_MATCHES[site_id].get("discharge_bocpd")
    m_s = ALL_MATCHES[site_id].get("static_discharge_p95")
    if m_b is None or m_s is None: continue
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"]).set_index("campaign_id")
    merged = m_b[["campaign_id","detected"]].rename(columns={"detected":"bocpd"}).merge(
        m_s[["campaign_id","detected"]].rename(columns={"detected":"static"}), on="campaign_id")
    categories = {"detected_by_both": merged[merged.bocpd & merged.static],
                   "bocpd_only": merged[merged.bocpd & ~merged.static],
                   "static_only": merged[~merged.bocpd & merged.static],
                   "missed_by_both": merged[~merged.bocpd & ~merged.static]}
    selected = {cat: df.iloc[0]["campaign_id"] for cat, df in categories.items() if len(df)}
    if not selected: continue

    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"]).set_index("datetime")
    instr_dis = STATION_INSTR[site_id].get("discharge")
    alarms_dis = ALL_ALARMS[site_id].get("discharge_bocpd")
    q95 = float(np.nanpercentile(iv["discharge_cfs"].dropna(), 95))

    fig, axes = plt.subplots(len(selected), 1, figsize=(11, 3.2*len(selected)), squeeze=False)
    for i, (cat, cid) in enumerate(selected.items()):
        ax = axes[i,0]
        c = campaigns.loc[cid]
        lo, hi = c["first_sample_time"]-pd.Timedelta(hours=24), c["last_sample_time"]+pd.Timedelta(hours=24)
        dis_win = iv["discharge_cfs"].loc[lo:hi]
        ax.plot(dis_win.index, dis_win.values, color=COLOR_DISCHARGE_BOCPD, lw=1.3, label="discharge")
        ax.axhline(q95, color=COLOR_STATIC_P95, ls="--", lw=1, label="static p95")
        ax.axvspan(c["first_sample_time"], c["last_sample_time"], color="gold", alpha=0.25, label="campaign window")
        if instr_dis is not None:
            ax2 = ax.twinx()
            surp_win = instr_dis[(instr_dis["datetime"]>=lo)&(instr_dis["datetime"]<=hi)]
            ax2.plot(surp_win["datetime"], surp_win["surprise"], color="grey", lw=0.9, alpha=0.8, label="surprise")
            ax2.set_ylabel("surprise", fontsize=7.5)
        if alarms_dis is not None:
            alarms_win = alarms_dis[(alarms_dis["alarm_time"]>=lo)&(alarms_dis["alarm_time"]<=hi)]
            for at in alarms_win["alarm_time"]:
                ax.axvline(at, color="#c1121f", lw=1.0, alpha=0.6)
        ax.set_title(f"{cat}: campaign {cid}", fontsize=9, loc="left")
        if i==0: ax.legend(fontsize=7, loc="upper left")
    fig.suptitle(f"{site_id}: representative case studies", fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_fig("D", f"fig18_case_studies_{site_id}.png", fig)


### Figure 19: Run-Length Posterior Diagnostics


In [ ]:
for site_id in EXTERNAL_STATIONS:
    instr_dis = STATION_INSTR[site_id].get("discharge")
    m_b = ALL_MATCHES[site_id].get("discharge_bocpd")
    if instr_dis is None or m_b is None: continue
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"]).set_index("campaign_id")
    detected_ids = m_b[m_b["detected"]==True]["campaign_id"]
    if len(detected_ids)==0: continue
    cid = detected_ids.iloc[0]
    c = campaigns.loc[cid]
    lo, hi = c["first_sample_time"]-pd.Timedelta(hours=24), c["last_sample_time"]+pd.Timedelta(hours=24)

    iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"]).set_index("datetime")
    win = instr_dis[(instr_dis["datetime"]>=lo)&(instr_dis["datetime"]<=hi)]
    dis_win = iv["discharge_cfs"].loc[lo:hi]

    fig, axes = plt.subplots(6, 1, figsize=(11,11), sharex=True)
    axes[0].plot(dis_win.index, dis_win.values, color=COLOR_DISCHARGE_BOCPD); axes[0].set_ylabel("discharge", fontsize=8)
    axes[1].plot(win["datetime"], win["surprise"], color="#888888"); axes[1].set_ylabel("surprise", fontsize=8)
    axes[2].plot(win["datetime"], win["cp_prob"], color="#7b2cbf"); axes[2].set_ylabel("P(r=0)", fontsize=8)
    axes[3].plot(win["datetime"], win["map_runlength"], color="#e07b39"); axes[3].set_ylabel("MAP\nrun length", fontsize=8)
    axes[4].plot(win["datetime"], win["map_drop"], color="#c1121f"); axes[4].set_ylabel("MAP\ndrop", fontsize=8)
    for k, col in zip([1,3,6], ["#1a6faf","#2a9d8f","#d62828"]):
        axes[5].plot(win["datetime"], win[f"short_run_mass_{k}"], color=col, label=f"short_run_mass_{k}", lw=1)
    axes[5].legend(fontsize=7, ncol=3); axes[5].set_ylabel("short-run\nmass", fontsize=8)
    for ax in axes: ax.axvspan(c["first_sample_time"], c["last_sample_time"], color="gold", alpha=0.2)
    fig.suptitle(f"{site_id}: run-length posterior diagnostics, campaign {cid}", fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_fig("D", f"fig19_runlength_diagnostics_{site_id}.png", fig)


### Figure 20: Score Distributions Near vs Away From Campaigns


In [ ]:
fig, axes = plt.subplots(len(EXTERNAL_STATIONS), 3, figsize=(13, 3*len(EXTERNAL_STATIONS)))
for i, site_id in enumerate(EXTERNAL_STATIONS):
    instr_dis = STATION_INSTR[site_id].get("discharge")
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"])
    if instr_dis is None: continue
    near = near_away_mask(instr_dis, campaigns, pad_h=6)
    for j, col in enumerate(["surprise","short_run_mass_3","cp_prob"]):
        ax = axes[i,j]
        bins = np.linspace(0, np.nanpercentile(instr_dis[col], 99.5), 60)
        ax.hist(instr_dis.loc[~near, col], bins=bins, density=True, alpha=0.5, color="grey", label="away")
        ax.hist(instr_dis.loc[near, col], bins=bins, density=True, alpha=0.6, color=COLOR_DISCHARGE_BOCPD, label="near campaign")
        if i==0: ax.set_title(col, fontsize=9)
        if j==0: ax.set_ylabel(site_id, fontsize=8.5)
        if i==0 and j==0: ax.legend(fontsize=7)
fig.suptitle("Score distributions, near vs away from campaigns (discharge BOCPD)", fontsize=11, y=1.0)
plt.tight_layout()
save_fig("D", "fig20_score_distributions.png", fig)


### Figure 21: Surprise Threshold Placement


In [ ]:
LOCKED = json.load(open(find_file(["makiki_04g_locked_config.json"], dir_=CONFIG_DIR, required=False) or
                          find_file(["makiki_04g_locked_config.csv"], dir_=TABLE_DIR, required=True)))["models"] \
    if find_file(["makiki_04g_locked_config.json"], dir_=CONFIG_DIR, required=False) else \
    {row["model"]: row.to_dict() for _, row in pd.read_csv(TABLE_DIR / "makiki_04g_locked_config.csv").iterrows()}
discharge_quantile = float(LOCKED["discharge_bocpd"]["quantile"])

fig, axes = plt.subplots(1, len(EXTERNAL_STATIONS), figsize=(5*len(EXTERNAL_STATIONS),5), sharey=True)
for i, site_id in enumerate(EXTERNAL_STATIONS):
    ax = axes[i]
    instr_dis = STATION_INSTR[site_id].get("discharge")
    if instr_dis is None: continue
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"])
    near = near_away_mask(instr_dis, campaigns, pad_h=6)
    threshold = float(instr_dis["surprise"].quantile(discharge_quantile))
    sorted_vals = np.sort(instr_dis["surprise"].values)
    ecdf = np.arange(1, len(sorted_vals)+1)/len(sorted_vals)
    ax.plot(sorted_vals, ecdf, color="black", lw=1)
    ax.axvline(threshold, color="#c1121f", ls="--", label=f"locked q={discharge_quantile}")
    ax.scatter(instr_dis.loc[near,"surprise"], np.full(near.sum(), 0.02), color=COLOR_DISCHARGE_BOCPD, s=4, alpha=0.3, label="near campaign")
    ax.set_title(site_id, fontsize=9); ax.set_xlabel("surprise")
    if i==0: ax.set_ylabel("ECDF"); ax.legend(fontsize=7)
    ax.set_xlim(0, np.nanpercentile(instr_dis["surprise"],99.5))
fig.suptitle("Surprise threshold placement vs empirical distribution", fontsize=11, y=1.02)
plt.tight_layout()
save_fig("D", "fig21_threshold_placement.png", fig)


### Figure 22: Score Separability by Station


In [ ]:
from scipy.stats import mannwhitneyu

def auroc_and_effect(near_vals, away_vals):
    near_vals = np.asarray(near_vals); away_vals = np.asarray(away_vals)
    if len(near_vals) < 2 or len(away_vals) < 2: return dict(auroc=np.nan, cohens_d=np.nan, rank_biserial=np.nan)
    U, _ = mannwhitneyu(near_vals, away_vals, alternative="two-sided")
    auroc = U / (len(near_vals)*len(away_vals))
    pooled_std = np.sqrt(((len(near_vals)-1)*near_vals.std(ddof=1)**2 + (len(away_vals)-1)*away_vals.std(ddof=1)**2) /
                           (len(near_vals)+len(away_vals)-2))
    cohens_d = (near_vals.mean()-away_vals.mean())/pooled_std if pooled_std>0 else np.nan
    return dict(auroc=round(auroc,4), cohens_d=round(cohens_d,4), rank_biserial=round(2*auroc-1,4))

sep_rows = []
for site_id in EXTERNAL_STATIONS:
    instr_dis = STATION_INSTR[site_id].get("discharge")
    if instr_dis is None: continue
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"])
    near = near_away_mask(instr_dis, campaigns, pad_h=6)
    for col in ["surprise","short_run_mass_3","cp_prob"]:
        stats = auroc_and_effect(instr_dis.loc[near,col], instr_dis.loc[~near,col])
        sep_rows.append(dict(site_id=site_id, score=col, **stats))
SEPARABILITY = pd.DataFrame(sep_rows)
print(SEPARABILITY.to_string(index=False))
if SAVE_OUTPUTS:
    save_csv(SEPARABILITY, FIG_DIRS["D"] / "fig22_score_separability.csv")

fig, ax = plt.subplots(figsize=(9,5.5))
for score, marker in zip(["surprise","short_run_mass_3","cp_prob"], ["o","s","^"]):
    sub = SEPARABILITY[SEPARABILITY["score"]==score]
    ax.plot(sub["site_id"], sub["auroc"], marker+"-", label=score)
ax.axhline(0.5, color="grey", ls=":", label="no separation")
ax.set_ylabel("AUROC (near vs away)"); ax.legend(fontsize=8.5)
ax.set_title("Score separability by station")
plt.tight_layout()
save_fig("D", "fig22_score_separability.png", fig)


### Figure 23: BOCPD Score Correlation Matrix

Uses changepoint probability, surprise, short-run mass, and MAP run-length drop.


In [ ]:
score_cols = ["cp_prob","surprise","short_run_mass_1","short_run_mass_3","short_run_mass_6","map_drop"]
fig, axes = plt.subplots(1, len(EXTERNAL_STATIONS), figsize=(5.5*len(EXTERNAL_STATIONS),5))
for i, site_id in enumerate(EXTERNAL_STATIONS):
    ax = axes[i]
    instr_dis = STATION_INSTR[site_id].get("discharge")
    if instr_dis is None: continue
    corr = instr_dis[score_cols].corr()
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(score_cols))); ax.set_xticklabels(score_cols, rotation=90, fontsize=6.5)
    ax.set_yticks(range(len(score_cols))); ax.set_yticklabels(score_cols, fontsize=6.5)
    for r in range(len(score_cols)):
        for c in range(len(score_cols)):
            ax.text(c, r, f"{corr.values[r,c]:.2f}", ha="center", va="center", fontsize=5.5)
    ax.set_title(site_id, fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.7, label="Pearson correlation")
fig.suptitle("BOCPD score correlation matrix (discharge): 'expected run-length drop' not saved, omitted", fontsize=10, y=1.05)
save_fig("D", "fig23_score_correlation.png", fig)


### Figure 24: Alarm-Score Event Trajectories


In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
rel_bins = np.arange(-24,24.5,1)  # Hourly bins from -24 to +24 h.
for site_id, color in zip(EXTERNAL_STATIONS, ["#1a6faf","#2a9d8f","#e07b39"]):
    instr_dis = STATION_INSTR[site_id].get("discharge")
    campaigns = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                              parse_dates=["first_sample_time","last_sample_time"])
    if instr_dis is None: continue
    dts = pd.DatetimeIndex(instr_dis["datetime"])
    all_rel = []
    for _, c in campaigns.iterrows():
        lo, hi = c["first_sample_time"]-pd.Timedelta(hours=24), c["first_sample_time"]+pd.Timedelta(hours=24)
        mask = (dts>=lo)&(dts<=hi)
        if mask.sum()==0: continue
        rel_h = (dts[mask]-c["first_sample_time"]).total_seconds()/3600
        vals = instr_dis.loc[mask,"surprise"].values
        all_rel.append(pd.DataFrame({"rel_h":rel_h, "surprise":vals, "campaign_id":c["campaign_id"]}))
    if not all_rel: continue
    combined = pd.concat(all_rel)
    combined["bin"] = pd.cut(combined["rel_h"], rel_bins)
    grouped = combined.groupby("bin")["surprise"].agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
    bin_centers = [b.mid for b in grouped.index]
    ax.plot(bin_centers, grouped["median"], color=color, label=site_id, lw=1.8)
    ax.fill_between(bin_centers, grouped.iloc[:,1], grouped.iloc[:,2], color=color, alpha=0.15)
ax.axvline(0, color="black", lw=1, alpha=0.5, label="field sample")
ax.set_xlabel("Hours relative to first field sample"); ax.set_ylabel("surprise (median, IQR band)")
ax.legend(fontsize=8.5)
ax.set_title("Event-aligned surprise trajectory")
plt.tight_layout()
save_fig("D", "fig24_event_trajectories.png", fig)


## Group E: Robustness and Transfer Diagnostics

Re-matches saved alarms or re-extracts saved scores. These checks do not rerun BOCPD.


In [ ]:
def static_alarms_generic(above_mask, persist_steps, merge_gap):
    arr = above_mask.fillna(False).values.astype(int)
    idx = above_mask.index
    diff = np.diff(np.concatenate([[0], arr, [0]]))
    starts = np.where(diff==1)[0]; ends = np.where(diff==-1)[0]-1
    episodes = [(idx[s], idx[e]) for s,e in zip(starts,ends) if (e-s+1) >= persist_steps]
    if not episodes: return pd.DataFrame(columns=["alarm_time","alarm_end"])
    merged = [episodes[0]]
    for s,e in episodes[1:]:
        if s - merged[-1][1] <= merge_gap: merged[-1] = (merged[-1][0], e)
        else: merged.append((s,e))
    return pd.DataFrame({"alarm_time":[m[0] for m in merged], "alarm_end":[m[1] for m in merged]})

def extract_alarms_episode_generic(score_df, score_col, threshold, merge_gap_h, persist_steps=2):
    above_mask = pd.Series(score_df[score_col].values >= threshold, index=pd.DatetimeIndex(score_df["datetime"]))
    above_mask = above_mask[~above_mask.index.duplicated(keep="last")].sort_index()
    ep = static_alarms_generic(above_mask, persist_steps, pd.Timedelta(hours=merge_gap_h))
    return ep[["alarm_time"]] if "alarm_time" in ep.columns else pd.DataFrame(columns=["alarm_time"])

def match_with_window_generic(alarms, before_h, after_h, campaigns):
    win_before = pd.Timedelta(hours=before_h); win_after = pd.Timedelta(hours=after_h)
    rows = []; matched_idx = set()
    for _, c in campaigns.iterrows():
        lo, hi = c["first_sample_time"]-win_before, c["last_sample_time"]+win_after
        in_win = alarms[(alarms["alarm_time"]>=lo)&(alarms["alarm_time"]<=hi)] if len(alarms) else pd.DataFrame()
        detected = len(in_win) > 0
        if detected: matched_idx.update(in_win.index.tolist())
        rows.append(dict(campaign_id=c["campaign_id"], detected=detected, n_alarms_in_window=len(in_win)))
    return pd.DataFrame(rows), matched_idx

print("Group E helpers defined.")


### Figures 25 & 26: Matching-Window and Alarm-Burden Sensitivity


In [ ]:
windows = [3, 6, 12, 24]
window_rows = []
for site_id in EXTERNAL_STATIONS:
    elig = pd.read_csv(PROCESSED_DIR / f"{site_id}_eligibility_audit.csv",
                         parse_dates=["first_sample_time","last_sample_time"])
    campaigns_full = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                                   parse_dates=["first_sample_time","last_sample_time"])
    for method, elig_col in [("discharge_bocpd","discharge_eligible"), ("static_discharge_p95","discharge_eligible")]:
        alarms = ALL_ALARMS[site_id].get(method)
        if alarms is None: continue
        elig_camps = campaigns_full[campaigns_full["campaign_id"].isin(elig.loc[elig[elig_col],"campaign_id"])]
        period_months = 1.0  
        iv = pd.read_csv(PROCESSED_DIR / f"{site_id}_iv_qc.csv", parse_dates=["datetime"])
        period_months = (iv["datetime"].max()-iv["datetime"].min()).days/30.44
        for w in windows:
            m, matched_idx = match_with_window_generic(alarms, w, w, elig_camps)
            n_det = int(m["detected"].sum()); n_tot = len(elig_camps)
            recall, _, _ = wilson_ci(n_det, n_tot)
            n_unmatched = len(alarms) - len(matched_idx)
            window_rows.append(dict(site_id=site_id, method=method, window_h=w, recall=round(recall,4),
                                      unmatched_per_month=round(n_unmatched/period_months,3)))
WINDOW_SENS = pd.DataFrame(window_rows)
if SAVE_OUTPUTS:
    save_csv(WINDOW_SENS, FIG_DIRS["E"] / "fig25_26_matching_window_sensitivity.csv")

fig, axes = plt.subplots(1, 2, figsize=(13,5.5))
for site_id, ls in zip(EXTERNAL_STATIONS, ["-","--",":"]):
    for method in ["discharge_bocpd","static_discharge_p95"]:
        sub = WINDOW_SENS[(WINDOW_SENS.site_id==site_id)&(WINDOW_SENS.method==method)]
        axes[0].plot(sub["window_h"], sub["recall"], ls, color=METHOD_COLORS[method], marker="o", ms=4,
                      label=f"{site_id[-4:]}/{method.split(chr(95))[0]}", alpha=0.8)
        axes[1].plot(sub["window_h"], sub["unmatched_per_month"], ls, color=METHOD_COLORS[method], marker="o", ms=4, alpha=0.8)
axes[0].set_xlabel("Matching window (+/- hours)"); axes[0].set_ylabel("Recall"); axes[0].legend(fontsize=6, ncol=2)
axes[0].set_title("Recall sensitivity (sensitivity only: not the primary result)")
axes[1].set_xlabel("Matching window (+/- hours)"); axes[1].set_ylabel("Unmatched alarms / month")
axes[1].set_title("Burden sensitivity")
plt.tight_layout()
save_fig("E", "fig25_26_matching_window_sensitivity.png", fig)


### Figure 27: Episode Merge-Gap Sensitivity

Varies the merge gap around the locked three-hour value without retuning it.


In [ ]:
merge_gaps = [0.5, 1.0, 2.0, 3.0, 4.0, 6.0, 9.0, 12.0]
locked_merge_gap = float(LOCKED["discharge_bocpd"].get("merge_gap_hours", 3.0))
locked_quantile = float(LOCKED["discharge_bocpd"]["quantile"])

mg_rows = []
for site_id in EXTERNAL_STATIONS:
    instr_dis = STATION_INSTR[site_id].get("discharge")
    elig = pd.read_csv(PROCESSED_DIR / f"{site_id}_eligibility_audit.csv",
                         parse_dates=["first_sample_time","last_sample_time"])
    campaigns_full = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                                   parse_dates=["first_sample_time","last_sample_time"])
    if instr_dis is None: continue
    elig_camps = campaigns_full[campaigns_full["campaign_id"].isin(elig.loc[elig["discharge_eligible"],"campaign_id"])]
    threshold = float(instr_dis["surprise"].quantile(locked_quantile))
    for mg in merge_gaps:
        alarms = extract_alarms_episode_generic(instr_dis, "surprise", threshold, mg)
        m, matched_idx = match_with_window_generic(alarms, 6, 6, elig_camps)
        n_det = int(m["detected"].sum())
        recall, _, _ = wilson_ci(n_det, len(elig_camps))
        dup = float(m.loc[m["detected"], "n_alarms_in_window"].mean()) if m["detected"].any() else np.nan
        mg_rows.append(dict(site_id=site_id, merge_gap_h=mg, recall=round(recall,4), duplicate_per_campaign=round(dup,3) if pd.notna(dup) else np.nan))
MG_SENS = pd.DataFrame(mg_rows)
if SAVE_OUTPUTS: save_csv(MG_SENS, FIG_DIRS["E"] / "fig27_mergegap_sensitivity.csv")

fig, axes = plt.subplots(1, 2, figsize=(12,5))
for site_id in EXTERNAL_STATIONS:
    sub = MG_SENS[MG_SENS.site_id==site_id]
    axes[0].plot(sub["merge_gap_h"], sub["recall"], "o-", label=site_id)
    axes[1].plot(sub["merge_gap_h"], sub["duplicate_per_campaign"], "o-", label=site_id)
for ax in axes: ax.axvline(locked_merge_gap, color="black", ls="--", alpha=0.6, label="locked")
axes[0].set_xlabel("Merge-gap (h)"); axes[0].set_ylabel("Recall"); axes[0].legend(fontsize=7.5)
axes[1].set_xlabel("Merge-gap (h)"); axes[1].set_ylabel("Duplicate alarms / campaign")
fig.suptitle("Episode merge-gap sensitivity (descriptive: not re-tuned)", fontsize=10.5)
plt.tight_layout()
save_fig("E", "fig27_mergegap_sensitivity.png", fig)


### Figure 28: Quantile Sensitivity Around the Locked Setting

Varies the score quantile around the locked setting without retuning it.


In [ ]:
quantiles = [0.95, 0.975, 0.99, 0.995, 0.999]
q_rows = []
for site_id in EXTERNAL_STATIONS:
    instr_dis = STATION_INSTR[site_id].get("discharge")
    elig = pd.read_csv(PROCESSED_DIR / f"{site_id}_eligibility_audit.csv",
                         parse_dates=["first_sample_time","last_sample_time"])
    campaigns_full = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                                   parse_dates=["first_sample_time","last_sample_time"])
    if instr_dis is None: continue
    elig_camps = campaigns_full[campaigns_full["campaign_id"].isin(elig.loc[elig["discharge_eligible"],"campaign_id"])]
    for q in quantiles:
        threshold = float(instr_dis["surprise"].quantile(q))
        alarms = extract_alarms_episode_generic(instr_dis, "surprise", threshold, locked_merge_gap)
        m, matched_idx = match_with_window_generic(alarms, 6, 6, elig_camps)
        n_det = int(m["detected"].sum())
        recall, _, _ = wilson_ci(n_det, len(elig_camps))
        period_months = (instr_dis["datetime"].max()-instr_dis["datetime"].min()).days/30.44
        unmatched_per_month = (len(alarms)-len(matched_idx))/period_months
        q_rows.append(dict(site_id=site_id, quantile=q, recall=round(recall,4), unmatched_per_month=round(unmatched_per_month,3)))
Q_SENS = pd.DataFrame(q_rows)
if SAVE_OUTPUTS: save_csv(Q_SENS, FIG_DIRS["E"] / "fig28_quantile_sensitivity.csv")

fig, axes = plt.subplots(1, 2, figsize=(12,5))
for site_id in EXTERNAL_STATIONS:
    sub = Q_SENS[Q_SENS.site_id==site_id]
    axes[0].plot(sub["quantile"], sub["recall"], "o-", label=site_id)
    axes[1].plot(sub["quantile"], sub["unmatched_per_month"], "o-", label=site_id)
for ax in axes: ax.axvline(locked_quantile, color="black", ls="--", alpha=0.6, label="locked")
axes[0].set_xlabel("Quantile"); axes[0].set_ylabel("Recall"); axes[0].legend(fontsize=7.5)
axes[1].set_xlabel("Quantile"); axes[1].set_ylabel("Unmatched alarms / month")
fig.suptitle("Quantile sensitivity around the locked setting (plateau vs cliff check)", fontsize=10.5)
plt.tight_layout()
save_fig("E", "fig28_quantile_sensitivity.png", fig)


### Figure 29: Transfer Degradation From Makiki


In [ ]:
metrics = ["recall","unmatched_alarms_per_month","matched_alarm_fraction","median_lead_lag_minutes"]
makiki_row = FINAL[(FINAL.site_id==MAKIKI_ID)&(FINAL.method=="discharge_bocpd")].iloc[0]
degr_rows = []
for site_id in EXTERNAL_STATIONS:
    ext_row = FINAL[(FINAL.site_id==site_id)&(FINAL.method=="discharge_bocpd")]
    if len(ext_row)==0: continue
    ext_row = ext_row.iloc[0]
    for met in metrics:
        degr_rows.append(dict(site_id=site_id, metric=met, delta=ext_row[met]-makiki_row[met]))
DEGR = pd.DataFrame(degr_rows)
if SAVE_OUTPUTS: save_csv(DEGR, FIG_DIRS["E"] / "fig29_transfer_degradation.csv")

fig, ax = plt.subplots(figsize=(9,5.5))
pivot = DEGR.pivot(index="site_id", columns="metric", values="delta")
pivot.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("External - Makiki (held-out)")
ax.set_title("Transfer degradation from Makiki, discharge BOCPD")
ax.legend(fontsize=7.5)
plt.tight_layout()
save_fig("E", "fig29_transfer_degradation.png", fig)


### Figure 30: Station-Level Rank Stability


In [ ]:
rank_metrics = ["recall","unmatched_alarms_per_month","matched_alarm_fraction"]
rank_rows = []
for site_id in STATION_ORDER:
    sub = FINAL[(FINAL.site_id==site_id) & (FINAL.method.isin(["discharge_bocpd","static_discharge_p95"]))]
    for met in rank_metrics:
        ascending = met == "unmatched_alarms_per_month"  # Lower burden ranks better.
        ranked = sub.set_index("method")[met].rank(ascending=ascending)
        for method, rank in ranked.items():
            rank_rows.append(dict(site_id=site_id, metric=met, method=method, rank=rank))
RANKS = pd.DataFrame(rank_rows)
if SAVE_OUTPUTS: save_csv(RANKS, FIG_DIRS["E"] / "fig30_rank_stability.csv")
print(RANKS.pivot_table(index=["site_id","metric"], columns="method", values="rank").to_string())


### Figure 31: Leave-One-Station-Out Descriptive Summary


In [ ]:
primary = FINAL[FINAL.role=="external_transfer_test"]
loso_rows = []
for excluded in EXTERNAL_STATIONS:
    remaining = primary[primary.site_id != excluded]
    for method in ["discharge_bocpd","static_discharge_p95"]:
        sub = remaining[remaining.method==method]
        n_elig = int(sub["n_eligible_campaigns"].sum())
        n_det = int((sub["recall"]*sub["n_eligible_campaigns"]).round().sum())
        loso_rows.append(dict(excluded_station=excluded, method=method, pooled_recall=round(n_det/n_elig,4) if n_elig else np.nan,
                                mean_unmatched_per_month=round(sub["unmatched_alarms_per_month"].mean(),3)))
LOSO = pd.DataFrame(loso_rows)
print(LOSO.to_string(index=False))
if SAVE_OUTPUTS: save_csv(LOSO, FIG_DIRS["E"] / "fig31_leave_one_station_out.csv")


### Figure 32: Campaign-Count Sensitivity (Bootstrap)


In [ ]:
N_BOOT_32 = 1000
boot_rows = []
for site_id in EXTERNAL_STATIONS:
    m_b = ALL_MATCHES[site_id].get("discharge_bocpd")
    m_s = ALL_MATCHES[site_id].get("static_discharge_p95")
    if m_b is None or m_s is None: continue
    merged = m_b[["campaign_id","detected"]].rename(columns={"detected":"bocpd"}).merge(
        m_s[["campaign_id","detected"]].rename(columns={"detected":"static"}), on="campaign_id")
    n = len(merged); idx_arr = np.arange(n)
    recalls_b, recalls_s = [], []
    for _ in range(N_BOOT_32):
        samp = merged.iloc[RNG.choice(idx_arr, n, replace=True)]
        recalls_b.append(samp["bocpd"].mean()); recalls_s.append(samp["static"].mean())
    boot_rows.append(dict(site_id=site_id, n_campaigns=n,
                            recall_bocpd_ci_lo=round(np.percentile(recalls_b,2.5),4), recall_bocpd_ci_hi=round(np.percentile(recalls_b,97.5),4),
                            recall_static_ci_lo=round(np.percentile(recalls_s,2.5),4), recall_static_ci_hi=round(np.percentile(recalls_s,97.5),4)))
BOOT32 = pd.DataFrame(boot_rows)
print(BOOT32.to_string(index=False))
if SAVE_OUTPUTS: save_csv(BOOT32, FIG_DIRS["E"] / "fig32_campaign_count_sensitivity.csv")

fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(BOOT32))
ax.errorbar(x-0.08, (BOOT32["recall_bocpd_ci_lo"]+BOOT32["recall_bocpd_ci_hi"])/2,
             yerr=(BOOT32["recall_bocpd_ci_hi"]-BOOT32["recall_bocpd_ci_lo"])/2, fmt="o", color=COLOR_DISCHARGE_BOCPD, capsize=4, label="discharge_bocpd")
ax.errorbar(x+0.08, (BOOT32["recall_static_ci_lo"]+BOOT32["recall_static_ci_hi"])/2,
             yerr=(BOOT32["recall_static_ci_hi"]-BOOT32["recall_static_ci_lo"])/2, fmt="s", color=COLOR_STATIC_P95, capsize=4, label="static_discharge_p95")
ax.set_xticks(x); ax.set_xticklabels([f"{s}\n(n={n})" for s,n in zip(BOOT32.site_id, BOOT32.n_campaigns)])
ax.set_ylabel("Bootstrap recall, 95% interval"); ax.legend(fontsize=8.5)
ax.set_title("Campaign-count bootstrap sensitivity")
plt.tight_layout()
save_fig("E", "fig32_campaign_count_sensitivity.png", fig)


### Figure 33: Coverage-Adjusted Performance


In [ ]:
summary = pd.read_csv(TABLE_DIR / "makiki_05a_cross_station_summary.csv", dtype={"site_id":str})
fig, axes = plt.subplots(1, 3, figsize=(15,5))
for site_id in EXTERNAL_STATIONS:
    row = FINAL[(FINAL.site_id==site_id)&(FINAL.method=="discharge_bocpd")]
    if len(row)==0: continue
    row = row.iloc[0]
    summ_row = summary[summary.site_id==site_id].iloc[0]
    axes[0].scatter(summ_row["discharge_coverage"], row["recall"], s=80); axes[0].annotate(site_id, (summ_row["discharge_coverage"], row["recall"]), fontsize=7)
    axes[1].scatter(row["n_eligible_campaigns"], row["recall"], s=80); axes[1].annotate(site_id, (row["n_eligible_campaigns"], row["recall"]), fontsize=7)
    axes[2].scatter(summ_row["discharge_coverage"], row["unmatched_alarms_per_month"], s=80); axes[2].annotate(site_id, (summ_row["discharge_coverage"], row["unmatched_alarms_per_month"]), fontsize=7)
axes[0].set_xlabel("Discharge coverage"); axes[0].set_ylabel("Recall")
axes[1].set_xlabel("N eligible campaigns"); axes[1].set_ylabel("Recall")
axes[2].set_xlabel("Discharge coverage"); axes[2].set_ylabel("Unmatched alarms/month")
fig.suptitle("Coverage-adjusted performance (n=3 external stations: sparse, descriptive only)", fontsize=10.5)
plt.tight_layout()
save_fig("E", "fig33_coverage_adjusted.png", fig)


## Group F: Static-Threshold and Turbidity-Specific Analyses


### Figure 34: Station-Specific Discharge Distributions and p95 Thresholds


In [ ]:
fig, axes = plt.subplots(1, len(ALL_STATION_IDS), figsize=(5*len(ALL_STATION_IDS),5), sharey=False)
for i, site_id in enumerate(ALL_STATION_IDS):
    ax = axes[i]
    dis = STATION_DIS_GRID[site_id].dropna()
    q95 = float(np.nanpercentile(dis, 95))
    diag = STATION_CAMPAIGN_DIAG[site_id]
    ax.hist(np.log1p(dis), bins=60, color="#cccccc", density=True)
    ax.axvline(np.log1p(q95), color=COLOR_STATIC_P95, ls="--", label="p95")
    ax.scatter(np.log1p(diag["peak_discharge_cfs"].dropna()), np.full(diag["peak_discharge_cfs"].notna().sum(), 0.02),
                color=COLOR_DISCHARGE_BOCPD, s=10, alpha=0.6, label="campaign peak")
    ax.set_title(site_id, fontsize=9); ax.set_xlabel("log1p(discharge cfs)")
    if i==0: ax.set_ylabel("density"); ax.legend(fontsize=7)
fig.suptitle("Discharge distribution, p95, and campaign sampling values", fontsize=11, y=1.03)
plt.tight_layout()
save_fig("F", "fig34_discharge_distributions.png", fig)


### Figure 35: Campaign Sampling Discharge Percentile


In [ ]:
fig, ax = plt.subplots(figsize=(8,5.5))
data = [STATION_CAMPAIGN_DIAG[s]["discharge_percentile"].dropna().values for s in ALL_STATION_IDS]
ax.boxplot(data, tick_labels=ALL_STATION_IDS, vert=True, widths=0.5)
for i, s in enumerate(ALL_STATION_IDS):
    vals = STATION_CAMPAIGN_DIAG[s]["discharge_percentile"].dropna().values
    ax.scatter(np.full(len(vals), i+1) + RNG.uniform(-0.1,0.1,len(vals)), vals, alpha=0.4, s=10, color="#1a6faf")
ax.set_ylabel("Discharge percentile at campaign sampling")
ax.set_title("Campaign sampling discharge percentile, by station")
plt.tight_layout()
save_fig("F", "fig35_sampling_percentile.png", fig)


### Figure 36: Fraction of Campaigns Already Above p90/p95/p99


In [ ]:
frac_rows = []
for site_id in ALL_STATION_IDS:
    pct = STATION_CAMPAIGN_DIAG[site_id]["discharge_percentile"].dropna()
    frac_rows.append(dict(site_id=site_id, frac_above_p90=round((pct>=90).mean(),4),
                            frac_above_p95=round((pct>=95).mean(),4), frac_above_p99=round((pct>=99).mean(),4)))
FRAC_ABOVE = pd.DataFrame(frac_rows)
print(FRAC_ABOVE.to_string(index=False))
if SAVE_OUTPUTS: save_csv(FRAC_ABOVE, FIG_DIRS["F"] / "fig36_frac_above_percentile.csv")

fig, ax = plt.subplots(figsize=(8.5,5.5))
x = np.arange(len(ALL_STATION_IDS)); w = 0.25
ax.bar(x-w, FRAC_ABOVE["frac_above_p90"], w, label="p90")
ax.bar(x, FRAC_ABOVE["frac_above_p95"], w, label="p95")
ax.bar(x+w, FRAC_ABOVE["frac_above_p99"], w, label="p99")
ax.set_xticks(x); ax.set_xticklabels(ALL_STATION_IDS)
ax.set_ylabel("Fraction of campaigns already above threshold"); ax.legend(fontsize=8.5)
ax.set_title("How strongly do field labels favor extreme flow? (is this Makiki-specific?)")
plt.tight_layout()
save_fig("F", "fig36_frac_above_percentile.png", fig)


### Figure 37: Static-Threshold Duration and Episode Burden


In [ ]:
ep_rows = []
for site_id in ALL_STATION_IDS:
    ep = STATION_STATIC_EPISODES[site_id]
    period_months = (STATION_DIS_GRID[site_id].index.max()-STATION_DIS_GRID[site_id].index.min()).days/30.44
    ep_rows.append(dict(site_id=site_id, n_episodes=len(ep),
                          median_duration_h=round(ep["duration_h"].median(),2) if len(ep) else np.nan,
                          total_active_h=round(ep["duration_h"].sum(),1) if len(ep) else 0,
                          episodes_per_month=round(len(ep)/period_months,2)))
EP_SUMMARY = pd.DataFrame(ep_rows)
print(EP_SUMMARY.to_string(index=False))
if SAVE_OUTPUTS: save_csv(EP_SUMMARY, FIG_DIRS["F"] / "fig37_static_episode_burden.csv")

fig, axes = plt.subplots(1, 3, figsize=(14,5))
axes[0].bar(EP_SUMMARY["site_id"], EP_SUMMARY["episodes_per_month"], color=COLOR_STATIC_P95)
axes[0].set_ylabel("Episodes / month")
axes[1].bar(EP_SUMMARY["site_id"], EP_SUMMARY["median_duration_h"], color=COLOR_STATIC_P95)
axes[1].set_ylabel("Median episode duration (h)")
axes[2].bar(EP_SUMMARY["site_id"], EP_SUMMARY["total_active_h"]/24, color=COLOR_STATIC_P95)
axes[2].set_ylabel("Total active time (days)")
fig.suptitle("Static p95 episode burden, full record", fontsize=11)
plt.tight_layout()
save_fig("F", "fig37_static_episode_burden.png", fig)


### Figure 38: Turbidity Availability vs Performance


In [ ]:
summary = pd.read_csv(TABLE_DIR / "makiki_05a_cross_station_summary.csv", dtype={"site_id":str})
fig, axes = plt.subplots(1, 2, figsize=(11,5))
for site_id in ALL_STATION_IDS:
    row = FINAL[(FINAL.site_id==site_id)&(FINAL.method=="turbidity_bocpd")]
    if len(row)==0: continue
    row = row.iloc[0]
    summ_row = summary[summary.site_id==site_id].iloc[0]
    axes[0].scatter(summ_row["turbidity_coverage"], row["recall"], s=80)
    axes[0].annotate(site_id, (summ_row["turbidity_coverage"], row["recall"]), fontsize=7)
    axes[1].scatter(row["n_eligible_campaigns"], row["unmatched_alarms_per_month"], s=80)
    axes[1].annotate(site_id, (row["n_eligible_campaigns"], row["unmatched_alarms_per_month"]), fontsize=7)
axes[0].set_xlabel("Turbidity coverage"); axes[0].set_ylabel("Turbidity BOCPD recall (secondary)")
axes[1].set_xlabel("N turbidity-eligible campaigns"); axes[1].set_ylabel("Unmatched alarms/month")
fig.suptitle("Turbidity availability vs performance (n=4 stations: sparse, descriptive)", fontsize=10)
plt.tight_layout()
save_fig("F", "fig38_turbidity_availability_vs_performance.png", fig)


### Figure 39: Turbidity Score Behavior Around Campaigns


In [ ]:
fig, axes = plt.subplots(len(EXTERNAL_STATIONS), 1, figsize=(9, 3.2*len(EXTERNAL_STATIONS)))
for i, site_id in enumerate(EXTERNAL_STATIONS):
    ax = axes[i] if len(EXTERNAL_STATIONS)>1 else axes
    instr_tur = STATION_INSTR[site_id].get("turbidity")
    elig = pd.read_csv(PROCESSED_DIR / f"{site_id}_eligibility_audit.csv",
                         parse_dates=["first_sample_time","last_sample_time"])
    campaigns_full = pd.read_csv(PROCESSED_DIR / f"{site_id}_storm_campaigns.csv",
                                   parse_dates=["first_sample_time","last_sample_time"])
    if instr_tur is None: continue
    tur_elig_camps = campaigns_full[campaigns_full["campaign_id"].isin(elig.loc[elig["turbidity_eligible"],"campaign_id"])]
    dts = pd.DatetimeIndex(instr_tur["datetime"])
    all_rel = []
    for _, c in tur_elig_camps.iterrows():
        lo, hi = c["first_sample_time"]-pd.Timedelta(hours=24), c["first_sample_time"]+pd.Timedelta(hours=24)
        mask = (dts>=lo)&(dts<=hi)
        if mask.sum()==0: continue
        rel_h = (dts[mask]-c["first_sample_time"]).total_seconds()/3600
        all_rel.append(pd.DataFrame({"rel_h":rel_h, "surprise":instr_tur.loc[mask,"surprise"].values}))
    if not all_rel: continue
    combined = pd.concat(all_rel)
    combined["bin"] = pd.cut(combined["rel_h"], np.arange(-24,24.5,2))
    grouped = combined.groupby("bin")["surprise"].median()
    bin_centers = [b.mid for b in grouped.index]
    ax.plot(bin_centers, grouped.values, color=COLOR_TURBIDITY_BOCPD, lw=2)
    ax.axvline(0, color="black", lw=1, alpha=0.5)
    ax.set_title(f"{site_id} (n={len(tur_elig_camps)} turbidity-eligible)", fontsize=9)
    ax.set_ylabel("median surprise")
axes[-1].set_xlabel("Hours relative to first field sample") if len(EXTERNAL_STATIONS)>1 else ax.set_xlabel("Hours relative to first field sample")
fig.suptitle("Turbidity surprise, event-aligned", fontsize=11, y=1.0)
plt.tight_layout()
save_fig("F", "fig39_turbidity_event_aligned.png", fig)


### Figure 40: Discharge vs Turbidity Alarm Agreement


In [ ]:
overlap_rows = []
for site_id in EXTERNAL_STATIONS:
    a_dis = ALL_ALARMS[site_id].get("discharge_bocpd")
    a_tur = ALL_ALARMS[site_id].get("turbidity_bocpd")
    if a_dis is None or a_tur is None: continue
    dis_times = pd.DatetimeIndex(a_dis["alarm_time"])
    tur_times = pd.DatetimeIndex(a_tur["alarm_time"])
    near_window = pd.Timedelta(hours=6)
    dis_matched = np.zeros(len(dis_times), dtype=bool)
    tur_matched = np.zeros(len(tur_times), dtype=bool)
    for i, t in enumerate(dis_times):
        if (np.abs(tur_times - t) <= near_window).any(): dis_matched[i] = True
    for i, t in enumerate(tur_times):
        if (np.abs(dis_times - t) <= near_window).any(): tur_matched[i] = True
    overlap_rows.append(dict(site_id=site_id, discharge_only=int((~dis_matched).sum()),
                               turbidity_only=int((~tur_matched).sum()),
                               near_coincident_pairs=int(dis_matched.sum())))
OVERLAP = pd.DataFrame(overlap_rows)
print(OVERLAP.to_string(index=False))
if SAVE_OUTPUTS: save_csv(OVERLAP, FIG_DIRS["F"] / "fig40_discharge_turbidity_agreement.csv")

fig, ax = plt.subplots(figsize=(8,5.5))
x = np.arange(len(OVERLAP)); w = 0.25
ax.bar(x-w, OVERLAP["discharge_only"], w, label="discharge-only", color=COLOR_DISCHARGE_BOCPD)
ax.bar(x, OVERLAP["near_coincident_pairs"], w, label="near-coincident (discharge side)", color="#7b2cbf")
ax.bar(x+w, OVERLAP["turbidity_only"], w, label="turbidity-only", color=COLOR_TURBIDITY_BOCPD)
ax.set_xticks(x); ax.set_xticklabels(OVERLAP["site_id"]); ax.legend(fontsize=7.5)
ax.set_ylabel("Alarm count")
ax.set_title("Discharge vs turbidity alarm agreement (+/-6h coincidence window)")
plt.tight_layout()
save_fig("F", "fig40_discharge_turbidity_agreement.png", fig)


## Group G: Summary tables

Collects the compact station and method summaries.


### Figure 41: Station-method performance


In [ ]:
dashboard_cols = ["n_eligible_campaigns","recall","ci_lo","ci_hi","unmatched_alarms_per_month",
                    "matched_alarm_fraction","duplicate_alarms_per_campaign","median_lead_lag_minutes"]
DASHBOARD = FINAL.set_index(["site_id","method"])[dashboard_cols].sort_index()
print(DASHBOARD.to_string())
if SAVE_OUTPUTS: save_csv(DASHBOARD.reset_index(), FIG_DIRS["G"] / "fig41_performance_dashboard.csv")

# Invert burden metrics so higher color values are better.


LOWER_IS_BETTER = {"unmatched_alarms_per_month", "duplicate_alarms_per_campaign"}
norm_data = (DASHBOARD - DASHBOARD.min()) / (DASHBOARD.max() - DASHBOARD.min())
for col in LOWER_IS_BETTER:
    if col in norm_data.columns: norm_data[col] = 1 - norm_data[col]

fig, ax = plt.subplots(figsize=(11, 0.45*len(DASHBOARD)+1))
im = ax.imshow(norm_data.values, aspect="auto", cmap="RdYlGn")
ax.set_yticks(range(len(DASHBOARD))); ax.set_yticklabels([f"{a}/{b}" for a,b in DASHBOARD.index], fontsize=7.5)
ax.set_xticks(range(len(dashboard_cols))); ax.set_xticklabels(dashboard_cols, rotation=45, ha="right", fontsize=7.5)
for r in range(len(DASHBOARD)):
    for c in range(len(dashboard_cols)):
        val = DASHBOARD.values[r,c]
        ax.text(c, r, f"{val:.2g}" if pd.notna(val) else "-", ha="center", va="center", fontsize=6)
ax.set_title("Full station x method performance dashboard\n(color: green=better, red=worse, per-column; burden columns inverted accordingly)")
plt.tight_layout()
save_fig("G", "fig41_performance_dashboard.png", fig)


### Figure 42: Inclusion criteria


In [ ]:
summary = pd.read_csv(TABLE_DIR / "makiki_05a_cross_station_summary.csv", dtype={"site_id":str})
MIN_ELIGIBLE = 10  # Descriptive sample-size threshold.
incl_rows = []
for _, row in summary.iterrows():
    for model, n_elig_col, cov_col in [("discharge_bocpd","n_discharge_eligible","discharge_coverage"),
                                          ("turbidity_bocpd","n_turbidity_eligible","turbidity_coverage"),
                                          ("joint_bocpd","n_joint_eligible",None)]:
        n_elig = row[n_elig_col]
        passes = n_elig >= MIN_ELIGIBLE
        reason = "" if passes else f"only {n_elig} eligible campaigns (< {MIN_ELIGIBLE})"
        incl_rows.append(dict(site_id=row["site_id"], model=model, n_eligible=int(n_elig),
                                coverage=round(row[cov_col],4) if cov_col else None,
                                passes_min_sample_size=passes, reason_if_excluded=reason))
INCLUSION = pd.DataFrame(incl_rows)
print(INCLUSION.to_string(index=False))
if SAVE_OUTPUTS: save_csv(INCLUSION, FIG_DIRS["G"] / "fig42_inclusion_criteria.csv")

fig, ax = plt.subplots(figsize=(9, 0.5*len(INCLUSION)+1))
colors_grid = INCLUSION["passes_min_sample_size"].map({True:"#2a9d8f", False:"#c1121f"}).values
ax.barh(range(len(INCLUSION)), INCLUSION["n_eligible"], color=colors_grid)
ax.axvline(MIN_ELIGIBLE, color="black", ls="--", label=f"descriptive min sample size ({MIN_ELIGIBLE})")
ax.set_yticks(range(len(INCLUSION))); ax.set_yticklabels([f"{r.site_id}/{r.model}" for _,r in INCLUSION.iterrows()], fontsize=7.5)
ax.set_xlabel("N eligible campaigns"); ax.legend(fontsize=8)
ax.set_title(f"Inclusion criteria dashboard (min-sample-size threshold of {MIN_ELIGIBLE} is descriptive, chosen for this figure only)")
plt.tight_layout()
save_fig("G", "fig42_inclusion_criteria.png", fig)


### Figure 43: Transfer criteria

Reports each transfer criterion separately; no composite score is used.


In [ ]:
verdict_rows = []
for site_id in EXTERNAL_STATIONS:
    bocpd_row = FINAL[(FINAL.site_id==site_id)&(FINAL.method=="discharge_bocpd")].iloc[0]
    static_row = FINAL[(FINAL.site_id==site_id)&(FINAL.method=="static_discharge_p95")].iloc[0]
    mg_sub = MG_SENS[MG_SENS.site_id==site_id]
    recall_at_static = bocpd_row["recall"] >= static_row["recall"]
    burden_no_worse = bocpd_row["unmatched_alarms_per_month"] <= static_row["unmatched_alarms_per_month"]
    earlier_detection = (bocpd_row["median_lead_lag_minutes"] < static_row["median_lead_lag_minutes"]) if pd.notna(static_row["median_lead_lag_minutes"]) else None
    stable_under_mg = bool((mg_sub["recall"].max() - mg_sub["recall"].min()) <= 0.10) if len(mg_sub) else None
    adequate_n = bocpd_row["n_eligible_campaigns"] >= MIN_ELIGIBLE
    verdict_rows.append(dict(site_id=site_id, recall_at_least_static=recall_at_static,
                               burden_no_worse_than_static=burden_no_worse, earlier_median_detection=earlier_detection,
                               stable_under_mergegap_sensitivity=stable_under_mg, adequate_sample_size=adequate_n))
VERDICT = pd.DataFrame(verdict_rows)
print("Transfer verdict matrix: each column is an independent named criterion. NOT combined into a composite score.")
print(VERDICT.to_string(index=False))
if SAVE_OUTPUTS: save_csv(VERDICT, FIG_DIRS["G"] / "fig43_transfer_verdict.csv")

fig, ax = plt.subplots(figsize=(9, 0.6*len(VERDICT)+1.5))
bool_cols = [c for c in VERDICT.columns if c != "site_id"]
def _bool_to_float(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return np.nan
    return 1.0 if v else 0.0
grid = VERDICT[bool_cols].apply(lambda col: col.map(_bool_to_float)).values
im = ax.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(VERDICT))); ax.set_yticklabels(VERDICT["site_id"])
ax.set_xticks(range(len(bool_cols))); ax.set_xticklabels(bool_cols, rotation=30, ha="right", fontsize=7.5)
for r in range(len(VERDICT)):
    for c in range(len(bool_cols)):
        v = grid[r,c]
        label = "PASS" if v==1 else ("FAIL" if v==0 else "n/a")
        ax.text(c, r, label, ha="center", va="center", fontsize=7)
ax.set_title("Transfer verdict: per-criterion, descriptive only (no composite score)")
plt.tight_layout()
save_fig("G", "fig43_transfer_verdict.png", fig)


## Summary


In [ ]:
print("="*78)
print("NOTEBOOK 05c: FIGURE INVENTORY SUMMARY")
print("="*78)
print(f"""
43 requested figures: 43 attempted, all produced something (figure and/or
CSV). Explicit, known limitations: not silently absorbed:

- Figure 1: drainage area/coordinates depend on station.csv coverage;
  confirmed present for all 4 stations this run.
- Figure 2: development/test split boundary marked for Makiki only --
  no holdout split was ever computed at the three external stations
  (05b applies the locked rule directly).
- Figure 23: "expected run-length drop" was never saved anywhere in this
  pipeline (only MAP run-length drop was): omitted from the
  correlation matrix, not substituted with something else.
- Figures 25-28: re-derived from already-saved alarms/scores, no BOCPD
  rerun: explicitly sensitivity/descriptive, never used to retune the
  locked external-station configuration.
- Figure 43: reports named criteria separately by design: no composite
  transfer-success score was invented.

All figures saved under Outputs/Figures/05c_inventory/{{A..G}}_*/
Table-equivalents saved alongside each figure (CSV) or in Outputs/Tables/.
""")
